In [ ]:
import glob
import os.path
from datetime import datetime
from itertools import chain
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from fabo import asset_root

In [ ]:
def annotation_interval_stats_ds_v0(limit_hi=60, limit_lo=0.25):
    timestamp_df = pd.DataFrame({"UTCタイムスタンプ": []}, dtype="object")
    
    # 収集対象である作成時間（UTCタイムスタンプ列）を含むJPEGファイル群。
    ds_v0_parquet_path_glob = chain(
        Path(asset_root()).glob("dataset/*/*/*.jpg"),
        Path(asset_root()).glob("detect/*/*/*.jpg"),
    )
    
    # 各JPEGファイルの作成時間（UTCタイムスタンプ列）を収集する。
    for f in ds_v0_parquet_path_glob:
        epoch = os.path.getmtime(f)
        timestamp_df = pd.concat([timestamp_df, pd.DataFrame({"UTCタイムスタンプ": [epoch]}, dtype="object")])

    # 昇順にソート
    timestamp_df = timestamp_df.sort_values(by="UTCタイムスタンプ")
    
    # 同時刻タイムスタンプは考慮外にする
    timestamp_df = timestamp_df.drop_duplicates()
    
    # 前後の作成時刻の差分をとり、作成インターバルを得る（最初の要素のNaNはドロップ）
    interval_df = timestamp_df.diff().dropna()
    
    # しきい値以上の作成インターバルを無視する
    interval_df = interval_df[(interval_df["UTCタイムスタンプ"] < limit_hi) & (interval_df["UTCタイムスタンプ"] > limit_lo)]
    
    return interval_df, timestamp_df


def annotation_interval_stats_ds_v1(limit_hi=60, limit_lo=0.25):
    timestamp_df = pd.DataFrame({"UTCタイムスタンプ": []}, dtype="object")
    
    # 収集対象である作成時間（UTCタイムスタンプ列）を含むParquetファイル群
    ds_v1_parquet_path_glob = chain(
        Path(asset_root()).glob("dataset/*/dataset_v1.parquet"),
        Path(asset_root()).glob("detect/*/dataset_v1.parquet"),
    )
    
    # 各データセット行の作成時間（UTCタイムスタンプ列）を収集する。
    # ただし、dataset/* 以下の値種別列が "y" の行は、
    # 値種別列が "x" のものに付随して作られる非推奨の情報なので収集対象から除外する。
    for f in ds_v1_parquet_path_glob:
        df = pd.read_parquet(f)
        if "値種別" in df:
            df = df[df["値種別"] != "y"]
        timestamp_df = pd.concat([timestamp_df, df["UTCタイムスタンプ"]])
    
    # 昇順にソート
    timestamp_df = timestamp_df.sort_values(by="UTCタイムスタンプ")
    
    # 同時刻タイムスタンプは考慮外にする
    timestamp_df = timestamp_df.drop_duplicates()
    
    # ISO 8601フォーマットからUNIX Epoch秒（浮動小数点あり）に変換する
    epoch_df = timestamp_df.map(lambda x: datetime.fromisoformat(x).timestamp())
    
    # 前後の作成時刻の差分をとり、作成インターバルを得る（最初の要素のNaNはドロップ）
    interval_df = epoch_df.diff().dropna()
    
    # しきい値以上の作成インターバルを無視する
    interval_df = interval_df[(interval_df["UTCタイムスタンプ"] < limit_hi) & (interval_df["UTCタイムスタンプ"] > limit_lo)]
    
    return interval_df, epoch_df


In [ ]:
interval_ds_v0_df, _ = annotation_interval_stats_ds_v0(limit_hi=30)

In [ ]:
interval_ds_v0_df

In [ ]:
inverval_ds_v1_df, _ = annotation_interval_stats_ds_v1(limit_hi=30)

In [ ]:
inverval_ds_v1_df

In [ ]:
# 作成インターバルのヒストグラムを表示
plt.figure(figsize=(10, 6))
plt.hist(interval_ds_v0_df, bins=120, edgecolor='black', alpha=0.7)
plt.title('Creation Interval Histogram', fontsize=16)
plt.xlabel('Creation Interval (s)', fontsize=14)
plt.ylabel('Frequency #', fontsize=14)
plt.grid(axis='y', alpha=0.5)

In [ ]:
# 作成インターバルのヒストグラムを表示
plt.figure(figsize=(10, 6))
plt.hist(inverval_ds_v1_df, bins=120, edgecolor='black', alpha=0.7)
plt.title('Creation Interval Histogram', fontsize=16)
plt.xlabel('Creation Interval (s)', fontsize=14)
plt.ylabel('Frequency #', fontsize=14)
plt.grid(axis='y', alpha=0.5)